<a href="https://colab.research.google.com/github/yasirdharejo786/Brain_Tumor_MRI_Using_Deep_Learning/blob/main/Research_Paper_Verification_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
HEC_FOLDER = "/content/drive/MyDrive/HEC"
pdf_files = [f for f in os.listdir(HEC_FOLDER) if f.endswith(".pdf")]
print(pdf_files)  # confirm sab files dikh rahi hain

['hjrs 2020-21.pdf', 'hjrs 2021-22.pdf', 'hjrs 2022-23.pdf', 'hjrs 2023-24.pdf', 'List of national journals 2024-25.pdf']


In [3]:
def find_header_map(table):
    """Detect column positions from the header row."""
    header_row = None
    for row in table[:3]:
        row_text = " ".join(str(c) for c in row if c).lower()
        if "title" in row_text or "journal" in row_text:
            header_row = row
            break
    if not header_row:
        return None

    col_map = {}
    for idx, cell in enumerate(header_row):
        if not cell:
            continue
        c = cell.lower().strip()
        if "title" in c or ("journal" in c and "name" in c):
            col_map["journal"] = idx
        elif "issn" in c:
            col_map["issn"] = idx
        elif "category" in c or "status" in c:
            col_map["category"] = idx
        elif "subject" in c and "sub" not in c:
            col_map["subject"] = idx
    return col_map

In [ ]:
import re, sqlite3, os

DB_PATH = "hjrs.db"

def find_header_and_map(table):
    """Find header row and map column names to indices."""
    for i, row in enumerate(table[:3]):
        row_text = " ".join(str(c) for c in row if c).lower()
        if "title" in row_text or ("journal" in row_text and "name" in row_text) or "issn" in row_text:
            col_map = {}
            for idx, cell in enumerate(row):
                if not cell:
                    continue
                c = cell.lower().strip()
                if "title" in c or "journal name" in c:
                    col_map["journal"] = idx
                elif "issn" in c:
                    col_map["issn"] = idx
                elif "categ" in c or "status" in c:
                    col_map["category"] = idx
                elif "subject area" in c or "discipline" in c:
                    col_map["subject"] = idx
                elif "country" in c:
                    col_map["country"] = idx
            return i, col_map
    return None, None


def split_issns(cell_text):
    """Extract one or more ISSNs from a cell (handles multi-line/combined cells)."""
    if not cell_text:
        return None, None
    found = re.findall(r"\d{4}-\d{3}[\dXx]", str(cell_text))
    issn_p = found[0] if len(found) > 0 else None
    issn_e = found[1] if len(found) > 1 else None
    return issn_p, issn_e


def extract_category(cell_text):
    """Handles both 'W' and 'Accepted for W' style category cells."""
    if not cell_text:
        return None
    text = str(cell_text).strip()
    match = re.search(r"\b([WXYZ])\b", text)
    return match.group(1) if match else None


def parse_pdf_file(path, period, list_type):
    import pdfplumber
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            print(f"    Processing page {page.page_number}...") # Added for debugging
            table = page.extract_table()
            if not table or len(table) < 2:
                continue
            header_idx, col_map = find_header_and_map(table)
            if not col_map or "journal" not in col_map:
                continue  # couldn't identify columns on this page, skip
            for row in table[header_idx + 1:]:
                if not row or len(row) <= max(col_map.values()):
                    continue
                journal = re.sub(r"\s+", " ", str(row[col_map.get("journal", -1)] or "")).strip()
                issn_p, issn_e = split_issns(row[col_map.get("issn")]) if "issn" in col_map else (None, None)
                category = extract_category(row[col_map.get("category")]) if "category" in col_map else None
                subject = str(row[col_map.get("subject")]).strip() if "subject" in col_map and row[col_map.get("subject")] else None
                country = str(row[col_map.get("country")]).strip() if "country" in col_map and row[col_map.get("country")] else None

                if not journal or not category:
                    continue
                yield (journal, issn_p, issn_e, category, subject, country, period, list_type)


def build_database(hec_folder):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS journals")
    cur.execute("""CREATE TABLE journals(
        journal_name TEXT, issn_p TEXT, issn_e TEXT,
        category TEXT, subject TEXT, country TEXT,
        period TEXT, list_type TEXT
    )""")

    total = 0
    for fname in os.listdir(hec_folder):
        if not fname.lower().endswith(".pdf"):
            continue
        path = os.path.join(hec_folder, fname)
        # crude auto-detect: filename tells us period + type
        list_type = "international" if "international" in fname.lower() else "national"
        period = fname.replace(".pdf", "")

        print(f"Parsing {fname} ({list_type})...")
        records_count = 0
        for r in parse_pdf_file(path, period, list_type):
            cur.execute("INSERT INTO journals VALUES (?,?,?,?,?,?,?,?)", r)
            records_count += 1
            total += 1
        print(f"  -> {records_count} records")

    conn.commit()
    conn.close()
    print(f"\nTotal inserted: {total}")


# ---- Run ----
!pip install pdfplumber -q
from google.colab import drive
drive.mount('/content/drive')

HEC_FOLDER = "/content/drive/MyDrive/HEC"
build_database(HEC_FOLDER)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Parsing hjrs 2020-21.pdf (national)...
    Processing page 1...
    Processing page 2...
    Processing page 3...
    Processing page 4...
    Processing page 5...
    Processing page 6...
    Processing page 7...
    Processing page 8...
    Processing page 9...
    Processing page 10...
    Processing page 11...
    Processing page 12...
    Processing page 13...
    Processing page 14...
    Processing page 15...
    Processing page 16...
    Processing page 17...
    Processing page 18...
    Processing page 19...
    Processing page 20...
    Processing page 21...
    Processing page 22...
    Processing page 23...
    Processing page 24...
    Processing page 25...
    Processing page 26...
    Processing page 27...
    Processing page 28...
    Processing page 29...
    Processing page 30...
    Processing page 31...
    Processing page 32...
    Proce